## What this notebook does

**GOAL: Create mapping between users, their datapoint contribution and inferred home census tract.**  

**WHEN: Use this notebook once per _region of interest (roi)_ and _period of interest (poi)_.**  

### Inputs
- `home-locations/US_yyyy_mm.csv`, from `extract_prep_US_level.ipynb`
- `datapoints/US_yyyy_mm.csv`, from `extract_prep_US_level.ipynb`

### Outputs
- `dpoints-homelocs/roi_yyyy-mm.csv`, roi level; all measured users with home location in NYC, with home location census tract and number of contributed datapoints

## Settings

In [ ]:
path = '/home/jovyan/Main/mobility-bias_K'

In [ ]:
# define outlier bounds #datapoints
min_dpoints = 30
max_dpoints = 100_000

In [ ]:
# region of interest
# roi = 'US'

# roi = 'NYC'
# counties = {'Bronx':'NY005', 'Kings':'NY047', 'New York':'NY061', 'Queens':'NY081', 'Richmond':'NY085'}

# roi = 'LA'
# counties = {'Los_Angeles':'CA037'} 

roi = 'Chicago'
counties = {'Cook':'IL031'} 

# roi = 'Houston'
# counties = {'Harris':'TX201'}

# roi = 'Phoenix'
# counties = {'Maricopa':'AZ013'}

# roi = 'Philadelphia'
# counties = {'Philadelphia':'PA101'}

# roi = 'San_Antonio'
# counties = {'Bexar':'TX029'}

# roi = 'San_Diego'
# counties = {'San_Diego':'CA073'}

# roi = 'Dallas'
# counties = {'Dallas':'TX113'}

# roi = 'San_Jose'
# counties = {'Santa_Clara':'CA085'}

In [ ]:
# period of interest
# April 2019 - pre COVID
period = '2019_04'
# Oct 2019
# period = '2019_10'

# April 2020 - during COVID lockdown/start lockdown
# period = '2020_04'
# Oct 2020
# period = '2020_10'

# April 2021 - late COVID
# period = '2021_04'
# Oct 2021
# period = '2021_10'

# April 2022 - post COVID
# period = '2022_04'
# Oct 2022
# period = '2022_10'

In [ ]:
file_name_import = 'US_' + period
file_name_export = roi + '_' + period

## Setup

In [ ]:
import os
os.chdir(path)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time
from bisect import insort

<a id='povtohome'></a>
## Map users to datapoint and home location data

#### Obtain home locations of users with non-outlier datapoints

In [ ]:
# load home_locations file for period of interest for whole US
home_locations = pd.read_csv('home-locations/' + file_name_import + '.csv')
# select users from home locations with home locations in NYC
home_locations = home_locations[home_locations.block_group_id.str.contains('|'.join(counties.values()))].reset_index(drop=True)
# replace block group id's by corresponding census tracts
home_locations['home_census_tract'] = home_locations['block_group_id'].map(lambda s: s[:-1])
home_locations = home_locations.drop('block_group_id', axis=1)
trcts_homelocs_start = len(home_locations.groupby('home_census_tract'))

# load datapoints file for period of interest for whole US
dpoints = pd.read_csv(f'datapoints/{file_name_import}.csv')
# select users from dpoints with home locations in NYC
dpoints = dpoints[dpoints['user_id'].isin(home_locations['user_id'])]
users_start = len(dpoints)

# overview start numbers (users and census tracts)
print(f'Total {roi} citizens in datapoints file: {users_start}\n')
print(f'Total {roi} citizens in home locations file: {len(home_locations)} (not important)')
print(f'Total {roi} census tracts in home locations file: {trcts_homelocs_start}\n')

In [ ]:
# remove users with outlier datapoint contribution
dpoints_new = dpoints[(dpoints['data_points'] >= min_dpoints) & (dpoints['data_points'] <= max_dpoints)]
outliers = len(dpoints) - len(dpoints_new)
rem_rate = outliers / len(dpoints) * 100

# overview of removal numbers of removing outliers
print(f'Removal outliers ({min_dpoints} < datapoints < {max_dpoints}):')
print(f'Removed {outliers} of {len(dpoints)} users from datapoints file')
print(f'Removal rate = {rem_rate:.2f}%\n')
print(f'{len(dpoints_new)} users left ({roi} citizens & not outliers)\n')
dpoints = dpoints_new
del dpoints_new

In [ ]:
# join datapoints and home locations data
dpoints_homelocs = home_locations.set_index('user_id').join(dpoints.set_index('user_id'), how='inner').reset_index(drop=False)
trcts_after_rem = len(dpoints_homelocs.groupby('home_census_tract'))

display(dpoints_homelocs)

In [ ]:
# save data
dpoints_homelocs.to_csv(f'dpoints_homelocs/{file_name_export}.csv', index=False)

In [ ]:
# save removal numbers
file = f'stats/{period}_removal_rates_outliers.txt'

newlines = [
    f'{file_name_export}',
    f' users_start={users_start} trcts_homelocs_start={trcts_homelocs_start}',
    f' rem_out{min_dpoints}_{max_dpoints}={outliers} rem_out_rate={rem_rate:.3f}%'
]

try: 
    with open(file) as f:
        data = f.read().splitlines()
except:
    data = []

if file_name_export in data:
    for i, newline in enumerate(newlines):
        data[data.index(file_name_export) + i] = newline
else:
    [data.insert(i, newline) for i, newline in enumerate(newlines)]

with open(file, 'w') as f:
    for l in data:
        f.write(l + '\n')